# Environment/config management (dotenv, pydantic-settings)

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### dotenv

> **Problem.** A developer pastes the OpenAI key into `config.py` to get a demo working and pushes the branch. GitHub's scanner flags it in minutes; before anyone reads the alert, an automated script has found it and run $400 of requests. The key was in the commit history even after the file was fixed.

**Idea.** Secrets live in a git-ignored `.env` locally and in the platform's secret store in production; code reads the same variable names either way.

**Use when** local development and CI.  
**Not when** production servers — the platform injects variables; there is no file.

```mermaid
flowchart LR
    E[".env (git-ignored)"] --> V[environment variables] --> C[os.environ]
    P["Kubernetes secret / · Secrets Manager"] --> V
```

**How it works.**
1. `.env` is a plain text file of `KEY=value` lines in the project folder, listed in `.gitignore` so it is never committed.
2. `find_dotenv()` searches upward from the current folder for it; `load_dotenv(path)` reads each line into `os.environ`.
3. Code reads `os.environ["OPENAI_API_KEY"]` — it does not know or care where the value came from.
4. In production there is no `.env`: Kubernetes, ECS or a secrets manager sets the same variable names before the process starts.
5. `dotenv_values(path)` reads a file into a dictionary without touching the environment — useful for tooling and tests.

| | what happens | result |
|:--|:--|:--|
| ✗ | key in source code | leaked on first push |
| ✓ | key in `.env` + `.gitignore` | loaded at runtime, printed as `sk-proj...1234` |

**Production code and its real output**

In [2]:
# dotenv — how secrets reach the process locally (.env, git-ignored). In production the same
# variable names are injected by the platform; the code does not change.
import os
import tempfile
from pathlib import Path

from dotenv import dotenv_values, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)
key = os.environ["OPENAI_API_KEY"]
print("OPENAI_API_KEY from .env:", key[:7] + "..." + key[-4:])

with tempfile.TemporaryDirectory() as folder:
    staging = Path(folder) / ".env.staging"
    staging.write_text("APP_ENV=staging\nLOG_LEVEL=DEBUG\n")
    print(".env.staging (read without touching os.environ):", dict(dotenv_values(staging)))
assert key.startswith("sk-")

OPENAI_API_KEY from .env: sk-proj...HH0A
.env.staging (read without touching os.environ): {'APP_ENV': 'staging', 'LOG_LEVEL': 'DEBUG'}


**What the output shows.** The key was found in the repo's `.env`, loaded, and printed masked; a second file (`.env.staging`) was read into a dictionary without changing the environment.

**In practice**
- **gitignore first** — add `.env` to `.gitignore` before creating the file; commit `.env.example` with placeholder values so others know what to set.
- **never in images** — a `.env` copied into a Docker image ships the secret to every registry and machine that pulls it.
- **rotate on leak** — a committed key is leaked the moment it is pushed; deleting the commit does not unpublish it — revoke and issue a new key.
- **override=False** — real environment variables win over `.env` by default; keep it that way so production values cannot be shadowed by a stray file.
- **one file per environment** — `.env.staging`, `.env.test` with `dotenv_values`, not one file with commented-out alternatives.

**Alternatives** — a secret manager SDK (Vault, AWS Secrets Manager) fetched at start-up · direnv for per-directory shells

**Terms** — *environment variable*: a named value the OS gives a program · *.env*: a local file of `KEY=value` lines


### pydantic-settings

> **Problem.** An operator sets `REQUEST_TIMEOUT_SECONDS=30s` (with the unit) in the deployment. The service starts fine; the value is only read when the first request comes in, where `float("30s")` throws. The service is up, health checks pass, and every request fails.

**Idea.** One validated Settings object built at start-up; bad configuration stops the program immediately, naming the field.

**Use when** every service.  
**Not when** never skip; even scripts benefit.

```mermaid
flowchart LR
    E[env / .env] --> S[Settings · types · ranges · secrets] -->|valid| R[service starts]
    S -->|invalid| X["exit: max_retries ≤ 5"]
```

**How it works.**
1. `class Settings(BaseSettings)` lists every setting with a type, a default and a rule: `max_retries: int = Field(default=2, ge=0, le=5)`.
2. `Settings()` reads matching environment variables (case-insensitive), then `.env`, converts and validates each value.
3. Any failure raises at construction — the first line of the program — with the field name and the rule that broke.
4. `SecretStr` wraps secrets: printing or logging the settings shows `**********`; `.get_secret_value()` gives the real string only where the client is built.
5. The one `settings` object is passed to whatever needs it (or imported from one module); nothing else reads `os.environ`.

| | what happens | result |
|:--|:--|:--|
| ✓ | `OPENAI_MODEL=gpt-4o` | overrides `.env` |
| ✗ | `MAX_RETRIES=99` | ValidationError at start-up |
| ✓ | `print(settings)` | `openai_api_key=SecretStr('**********')` |

**Production code and its real output**

In [3]:
# pydantic-settings — the Settings object the notebook started with. Secrets are masked, numbers
# validated, the environment overrides .env, and bad config fails at startup instead of mid-request.
import os

from pydantic import ValidationError

show("settings (secret masked)", settings.model_dump())

os.environ["OPENAI_MODEL"] = "gpt-4o"
print("environment override ->", Settings().openai_model)
del os.environ["OPENAI_MODEL"]

os.environ["MAX_RETRIES"] = "99"
try:
    Settings()
except ValidationError as error:
    print("MAX_RETRIES=99 ->", error.errors()[0]["msg"])
    startup_error = error.errors()[0]["msg"]
del os.environ["MAX_RETRIES"]
assert "less than or equal to 5" in startup_error

settings (secret masked)
{
  "openai_api_key": "**********",
  "openai_model": "gpt-4o-mini",
  "request_timeout_seconds": 30.0,
  "max_retries": 2
}
environment override -> gpt-4o
MAX_RETRIES=99 -> Input should be less than or equal to 5


**What the output shows.** The settings printed with the key masked; setting `OPENAI_MODEL` in the environment overrode the file; `MAX_RETRIES=99` was rejected before anything ran, with the rule (`≤ 5`) in the message.

**In practice**
- **build once, inject** — construct `settings` at start-up and pass it in; `os.environ` reads scattered in request code are unvalidated and untestable.
- **ranges everywhere** — `ge`/`le`/`gt` on every number catch typos and unit mistakes at deploy time instead of at 3 a.m.
- **SecretStr always** — keys, tokens and passwords as `SecretStr`, so a stray `print(settings)` or a log line cannot leak them.
- **safe defaults** — a default must be safe in production, not just convenient on a laptop (`debug=False`, `timeout=30`, not `timeout=None`).
- **print at boot** — log the masked settings once at start-up; when something is misconfigured, the first thing you want is to see what the process actually loaded.

**Alternatives** — `os.environ` reads scattered in code (no validation, no docs) · Dynaconf (layered config files) · Hydra for ML experiment configs

**Terms** — *SecretStr*: a value that prints as `**********` · *fail fast*: crash at start-up rather than mid-request
